# Training Runs — Montezuma's Revenge

Quick-launch notebook. Run the **Helpers** cell first, then any run cell.

- Smoke test runs synchronously (~3 min, shows output inline).
- All other cells launch in the background — they return immediately and print the log path.
- Videos → `~/work/montezuma/videos/<run_name>/` · Logs → `~/work/montezuma/logs/`

> **Path note:** the repo lives in `~/work/montezuma` (not `~/montezuma`).
> Files outside `~/work/` are deleted when the JupyterHub server stops.

## Helpers

Run once per session before anything else.

In [ ]:
import subprocess, sys, pathlib

# ~/work/ persists across JupyterHub restarts; ~/  does not
PROJECT = pathlib.Path.home() / "work" / "montezuma"
for d in ["logs", "videos"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)

def launch(args, log_name):
    """Start a training script in the background. Returns the Popen handle."""
    log_path = PROJECT / "logs" / log_name
    proc = subprocess.Popen(
        [sys.executable, *args],
        stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
        cwd=PROJECT,
    )
    print(f"PID {proc.pid}  →  {log_path}")
    return proc

print("Ready. PROJECT =", PROJECT)

---

## Smoke test — RND, 50 k steps (~3 min)

Runs synchronously and shows output inline. Confirms GPU + ALE + training loop.
Expected: `Using device: cuda`, SPS > 1 000.

In [ ]:
result = subprocess.run(
    [
        sys.executable, "src/agents/rnd.py",
        "--total-timesteps", "50000",
        "--num-envs", "8",
        "--obs-norm-init-steps", "2",
        "--checkpoint-interval", "999",
        "--runs-dir", "/tmp/smoke-runs",
        "--sync-envs",
    ],
    capture_output=True, text=True, cwd=PROJECT,
)
print(result.stdout[-3000:] if result.stdout else "(no output)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

---

## PPO test run — 200 k steps (~10–15 min)

Verifies the full pipeline end-to-end: training, `RecordVideo`, TensorBoard logs.
Records every 10th episode → ~10–20 clips in `videos/`.

In [ ]:
ppo_test = launch([
    "src/agents/ppo.py",
    "--total-timesteps", "200000",
    "--num-envs", "4",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
    "--video-episode-interval", "10",
], "ppo_test.out")

---

## Count-based test run — 200 k steps (~10–15 min)

PPO + SimHash exploration bonus. First algorithm in the thesis roadmap.

In [ ]:
count_test = launch([
    "src/agents/count_based.py",
    "--total-timesteps", "200000",
    "--num-envs", "4",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
], "count_test.out")

---

## RND test run — 200 k steps (~10–15 min)

PPO + Random Network Distillation. Key algorithm of the thesis.

In [ ]:
rnd_test = launch([
    "src/agents/rnd.py",
    "--total-timesteps", "200000",
    "--num-envs", "4",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
], "rnd_test.out")

---

## Full production runs

These take 1–10 h on a V100. Add `--track` to monitor via W&B without keeping the browser open.
Adjust `--seed` to run multiple seeds.

In [ ]:
# PPO — 5 M steps
ppo_full = launch([
    "src/agents/ppo.py",
    "--total-timesteps", "5000000",
    "--num-envs", "8",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
    "--video-episode-interval", "50",
    "--track",
], "ppo_full_s1.out")

In [ ]:
# Count-based — 5 M steps
count_full = launch([
    "src/agents/count_based.py",
    "--total-timesteps", "5000000",
    "--num-envs", "8",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
    "--track",
], "count_full_s1.out")

In [ ]:
# RND — 50 M steps
rnd_full = launch([
    "src/agents/rnd.py",
    "--total-timesteps", "50000000",
    "--num-envs", "8",
    "--seed", "1",
    "--sync-envs",
    "--capture-video",
    "--track",
], "rnd_full_s1.out")

---

## Monitor

Tail a log or check whether launched processes are still running.

In [ ]:
# Tail the last 60 lines of a log
log_name = "ppo_test.out"  # ← change as needed
result = subprocess.run(
    ["tail", "-n", "60", str(PROJECT / "logs" / log_name)],
    capture_output=True, text=True,
)
print(result.stdout or "(log is empty)")

In [ ]:
# Check status of all launched processes
for label in ["ppo_test", "count_test", "rnd_test", "ppo_full", "count_full", "rnd_full"]:
    proc = globals().get(label)
    if proc is None:
        print(f"{label}: not started")
    elif proc.poll() is None:
        print(f"{label}: running  (PID {proc.pid})")
    else:
        print(f"{label}: finished (exit code {proc.poll()})")